# 02. 분야를 좁힌 캡셔닝 (포켓몬)
포켓몬 캡션으로 미세조정된 `kr-manish/git-base-pokemon`과 일반 모델 `git-base-coco`를 같은 그림에 돌려 비교합니다. 그림은 `python prepare_data.py`로 받습니다.

주의: 시험 그림은 미세조정에 쓰인 데이터셋에서 가져온 것이라, 모델이 학습 때 이미 본 그림일 수 있습니다.

In [1]:
import os, time, warnings
warnings.filterwarnings("ignore")
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM
device = "cpu"

In [2]:
import json
processor = AutoProcessor.from_pretrained("microsoft/git-base")
model = AutoModelForCausalLM.from_pretrained("kr-manish/git-base-pokemon").to(device)
general_processor = AutoProcessor.from_pretrained("microsoft/git-base-coco")
general = AutoModelForCausalLM.from_pretrained("microsoft/git-base-coco").to(device)
def caption(model, processor, image):
    inputs = processor(images=image, return_tensors="pt").to(device)
    generated_ids = model.generate(pixel_values=inputs.pixel_values, max_length=50)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
truth = {m["file"]: m["text"] for m in json.load(open("../data/pokemon/captions.json"))}

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [3]:
for f in ["test_000.jpg", "test_001.jpg", "test_002.jpg", "test_003.jpg", "test_005.jpg"]:
    image = Image.open(f"../data/pokemon/{f}").convert("RGB")
    print(f"[{f}]\n  데이터셋 캡션 : {truth[f]}\n  포켓몬 모델   : {caption(model, processor, image)}"
          f"\n  일반 모델     : {caption(general, general_processor, image)}")

[test_000.jpg]
  데이터셋 캡션 : a yellow and white cartoon character with a red eye
  포켓몬 모델   : a yellow and white pokemon character with a red eye
  일반 모델     : a cartoon of a cute cartoon character.


[test_001.jpg]
  데이터셋 캡션 : a black and white bird with a red beak
  포켓몬 모델   : a bird with a yellow beak and black wings
  일반 모델     : a cartoon of a bird.


[test_002.jpg]
  데이터셋 캡션 : a drawing of a pokemon with a green leaf on it's back
  포켓몬 모델   : a green and yellow cartoon character with a green leaf
  일반 모델     : a cartoon of a dragon with a green scarf.


[test_003.jpg]
  데이터셋 캡션 : a cartoon of a blue and black dragon
  포켓몬 모델   : a drawing of a blue and black dragon
  일반 모델     : a dragon from the movie


[test_005.jpg]
  데이터셋 캡션 : a white and purple cartoon character with green eyes
  포켓몬 모델   : a drawing of a purple and white animal
  일반 모델     : a cartoon of a rabbit


### 반대로: 포켓몬 모델에 일반 사진을 넣으면?

In [4]:
for f in ["96973080_783e375945.jpg", "99679241_adc853a5c0.jpg", "979383193_0a542a059d.jpg"]:
    image = Image.open(f"../data/photos/{f}").convert("RGB")
    print(f"{f:28s} 포켓몬 모델 -> {caption(model, processor, image)}")

96973080_783e375945.jpg      포켓몬 모델 -> a drawing of a tree with a leaf on it


99679241_adc853a5c0.jpg      포켓몬 모델 -> a bird with its head on a white surface


979383193_0a542a059d.jpg     포켓몬 모델 -> a group of people with their arms around each other
